<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# -------------------------------------------------
# 1. Input data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)

# Read the CSV file uploaded to Google Colab
data = pd.read_csv('/content/GTPP-DATASET.csv')

# 8 input features
X = data[[
    'AT',
    'AP',
    'DF',
    'CH',
    'GP',
    'CPR',
    'CPD',
    'TTXM'
]].values

# Target Output
y = data['EP'].values


# -------------------------------------------------
# 2. Train-Test Split (80:20 Ratio)
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# -------------------------------------------------
# 3. MinMax Scaling
# -------------------------------------------------

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -------------------------------------------------
# 4. Phase 1 Experiments (Varying Learning Rate & Batch Size)
# -------------------------------------------------

# We pick 128 neurons (instead of 1500) to keep training fast and prevent overfitting
NEURONS = 128
EPOCHS = 50

# List of (learning_rate, batch_size) combinations to test
configs_to_test = [
    (0.001, 32),   # Baseline (Default Adam learning rate & default batch size)
    (0.001, 16),   # Smaller batch size
    (0.001, 64),   # Larger batch size
    (0.0005, 32),  # Smaller learning rate
    (0.002, 32)   # Larger learning rate
]

# List to store final results for each experiment
results_list = []

print("Starting Phase 1 Experiments...\n")

for lr, bs in configs_to_test:
    print(f"Testing Learning Rate = {lr} | Batch Size = {bs} ...")

    # Reset random seed before building each model for fair comparison
    tf.keras.utils.set_random_seed(42)

    # Build Model
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(NEURONS, activation='relu'),
        tf.keras.layers.Dense(1)
    ])

    # Compile Model with current learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )

    # Train Model (verbose=0 keeps output clean without flooding screen)
    model.fit(
        X_train_scaled,
        y_train,
        epochs=EPOCHS,
        batch_size=bs,
        validation_split=0.2,
        verbose=0
    )

    # Predict on Test Data
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    # Calculate Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Save results in a dictionary
    results_list.append({
        "Learning Rate": lr,
        "Batch Size": bs,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })


# -------------------------------------------------
# 5. Display Phase 1 Comparison Results
# -------------------------------------------------

# Convert results into a clean Pandas Table
results_df = pd.DataFrame(results_list)

print("\n==================================================")
print("            PHASE 1 EXPERIMENT RESULTS            ")
print("==================================================")
print(results_df.to_string(index=False))

Starting Phase 1 Experiments...

Testing Learning Rate = 0.001 | Batch Size = 32 ...
Testing Learning Rate = 0.001 | Batch Size = 16 ...
Testing Learning Rate = 0.001 | Batch Size = 64 ...
Testing Learning Rate = 0.0005 | Batch Size = 32 ...
Testing Learning Rate = 0.002 | Batch Size = 32 ...

            PHASE 1 EXPERIMENT RESULTS            
 Learning Rate  Batch Size      MAE       MSE     RMSE  R2 Score
        0.0010          32 1.488119  3.569445 1.889297  0.909055
        0.0010          16 1.347490  2.955945 1.719286  0.924687
        0.0010          64 2.949629 14.303516 3.781999  0.635566
        0.0005          32 2.476053  9.901189 3.146615  0.747731
        0.0020          32 1.335412  2.958588 1.720055  0.924619
